# Forecast Bridge: Bottom-Up vs Top-Down

## Inputs
- **Bottom-up CSV** — export from the main demand planning tool:
  `composite_primary_key | product_name | code | customers__customer_category | month_num | month_year | units`
- **Top-down CSV** — simple 3-column file:
  `SKU | Month | Top` (no channel — bridge splits across channels + categories statistically)

## What it produces
| Tab | Contents |
|-----|----------|
| `Share Stability` | CV, trend, structural break diagnostics per SKU × channel × category |
| `Bridge` | Side-by-side bottom-up vs top-down vs difference per SKU × channel × month |
| `Export - Top-Down Split` | Upload-ready rows matching the bottom-up format exactly |

## Allocation logic
| Status | Rule |
|--------|------|
| **Stable** | Historical mean share |
| **Monitor** | 50/50 blend of historical mean + equal weight |
| **Unstable** | Pure equal weight |


In [ ]:
!pip install gspread oauth2client pandas numpy scipy -q

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
from google.colab import files, auth
import gspread
from google.auth import default
print("\u2705 Libraries loaded")

✅ Libraries loaded


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
CV_STABLE_MAX        = 0.25   # CV below this → Stable
CV_MONITOR_MAX       = 0.50   # CV below this → Monitor; above → Unstable
TREND_PVALUE_MAX     = 0.10   # p-value threshold for significant trend
TREND_SLOPE_MIN      = 0.005  # minimum slope (share units/month) to matter
BREAK_DIFF_MAX       = 0.10   # max |recent_mean − prior_mean| before flagging
BREAK_RECENT_MONTHS  = 6      # months counted as "recent"
MONITOR_BLEND_WEIGHT = 0.5    # 0 = full equal-weight, 1 = full historical mean
SHARE_MONTHS         = 12     # history window for share calculation
SPLIT_CUTOFF         = '2026-02'
EXPORT_CHANNELS      = ['Direct-to-Consumer', 'Wholesale']

MONTH_NAMES = {
    '01':'January','02':'February','03':'March','04':'April',
    '05':'May',    '06':'June',   '07':'July', '08':'August',
    '09':'September','10':'October','11':'November','12':'December'
}
MONTH_ABBR = {v[:3].lower(): k for k, v in MONTH_NAMES.items()}

def parse_period(s):
    """Robustly parse any month string to a pd.Period."""
    s = str(s).strip()
    if not s: return None
    # Skip bare month names (e.g. "January") from hitting pd.Period directly
    # — pandas parses them as year=1 which is wrong
    if not re.search(r'\d', s):
        pass  # no digits — go straight to regex fallback with year injection
    else:
        try: return pd.Period(s, freq='M')
        except Exception: pass
    # 'Jan', 'Jan-26', 'Jan 26', 'January-2026', etc.
    m = re.match(r'^([A-Za-z]{3,9})[- \s]?(\d{0,4})$', s)
    if m:
        mon = m.group(1)[:3].lower()
        yr  = m.group(2)
        mn  = MONTH_ABBR.get(mon)
        if mn:
            if   len(yr) == 2: yr = ('20' if int(yr)<50 else '19') + yr
            elif len(yr) == 4: pass
            else:              yr = str(pd.Period(SPLIT_CUTOFF, freq='M').year)
            try: return pd.Period(f"{yr}-{mn}", freq='M')
            except Exception: pass
    return None

def period_to_my(p):
    """pd.Period → 'January 2026' style label."""
    return f"{MONTH_NAMES[f'{p.month:02d}']} {p.year}"

print("Config loaded")

Config loaded


In [ ]:
# ── Upload bottom-up forecast CSV ────────────────────────────────────────
# Format: composite_primary_key | product_name | code |
#         customers__customer_category | month_num | month_year | units
print("Upload BOTTOM-UP forecast CSV (export from the main demand planning tool):")
up_bu = files.upload()
bu_filename = list(up_bu.keys())[0]
print(f"Loaded: {bu_filename}")

Upload BOTTOM-UP forecast CSV (export from the main demand planning tool):


Saving Demand_Planning_v3 - Export (1).csv to Demand_Planning_v3 - Export (1).csv
Loaded: Demand_Planning_v3 - Export (1).csv


In [ ]:
# ── Upload sales history CSV ─────────────────────────────────────────────
# Same file used in the main demand planning tool
print("Upload SALES HISTORY CSV:")
up_sales = files.upload()
sales_filename = list(up_sales.keys())[0]
print(f"Loaded: {sales_filename}")

Upload SALES HISTORY CSV:


Saving _SELECT_mts_created_date_EXTRACT_YEAR_FROM_mts_created_date_100__202603031704.csv to _SELECT_mts_created_date_EXTRACT_YEAR_FROM_mts_created_date_100__202603031704.csv
Loaded: _SELECT_mts_created_date_EXTRACT_YEAR_FROM_mts_created_date_100__202603031704.csv


In [ ]:
# ── Upload top-down forecast CSV ─────────────────────────────────────────
# Format: SKU | Month | Top   (month = Jan, Jan-26, 2026-01, etc.)
print("Upload TOP-DOWN forecast CSV (SKU | Month | Top):")
up_td = files.upload()
td_filename = list(up_td.keys())[0]
print(f"Loaded: {td_filename}")

Upload TOP-DOWN forecast CSV (SKU | Month | Top):


Saving Demand_Planning_v3 - Exec (3).csv to Demand_Planning_v3 - Exec (3).csv
Loaded: Demand_Planning_v3 - Exec (3).csv


In [ ]:
# ============================================================
# PARSE BOTTOM-UP
# Actual format: composite_primary_key | product_name | code |
#                channel | month_num | month_year | units
# NOTE: customers__customer_category is NOT in this file —
#       the bottom-up is already aggregated to SKU × channel × month.
#       The bridge will compare at that grain.
# ============================================================
bu_raw = pd.read_csv(bu_filename)
print(f"Bottom-up raw: {len(bu_raw):,} rows")
print(f"Columns: {list(bu_raw.columns)}")
print(bu_raw.head(3).to_string())

bu = bu_raw.copy()

# Rename to internal names — channel column exists directly
bu = bu.rename(columns={
    'code':    'sku',
    'units':   'bu_units',
    # 'channel' stays as 'channel'
})

# Verify channel column exists
if 'channel' not in bu.columns:
    # Fall back: extract from composite_primary_key
    def extract_channel(key):
        for ch in EXPORT_CHANNELS:
            if f'-{ch}-' in str(key):
                return ch
        return 'Unknown'
    bu['channel'] = bu['composite_primary_key'].apply(extract_channel)
    print("  channel derived from composite_primary_key")
else:
    print(f"  channel column found: {bu['channel'].unique()}")

# Parse month_year → Period  (e.g. 'Feb-26')
bu['period']    = bu['month_year'].apply(parse_period)
bu              = bu.dropna(subset=['period'])
bu['month_str'] = bu['period'].astype(str)   # 'YYYY-MM'
bu['bu_units']  = pd.to_numeric(bu['bu_units'], errors='coerce').fillna(0).astype(int)

# Keep only known channels
bu = bu[bu['channel'].isin(EXPORT_CHANNELS)].copy()

# Build name lookup
name_lookup = dict(zip(bu['sku'], bu['product_name']))

print(f"\nBottom-up parsed: {len(bu):,} rows")
print(f"  SKUs     : {bu['sku'].nunique()}")
print(f"  Channels : {bu['channel'].unique()}")
print(f"  Months   : {sorted(bu['month_str'].unique())}")
print(f"  Total units: {bu['bu_units'].sum():,}")
print()
print("Sample:")
print(bu[['sku','channel','month_year','bu_units']].head(4).to_string(index=False))


Bottom-up raw: 1,999 rows
Columns: ['composite_primary_key', 'product_name', 'code', 'channel', 'customers__customer_category', 'month_num', 'month_year', 'units']
                                                           composite_primary_key                         product_name      code             channel customers__customer_category  month_num month_year  units
0   Balancing Hypotonic - Retail (4 oz)-FG-10004-Direct-to-Consumer-January 2026  Balancing Hypotonic - Retail (4 oz)  FG-10004  Direct-to-Consumer           Direct-to-Consumer     202601     Jan-26    221
1  Balancing Hypotonic - Retail (4 oz)-FG-10004-Direct-to-Consumer-February 2026  Balancing Hypotonic - Retail (4 oz)  FG-10004  Direct-to-Consumer           Direct-to-Consumer     202602     Feb-26    217
2     Balancing Hypotonic - Retail (4 oz)-FG-10004-Direct-to-Consumer-March 2026  Balancing Hypotonic - Retail (4 oz)  FG-10004  Direct-to-Consumer           Direct-to-Consumer     202603     Mar-26    385
  channel co

In [ ]:
# ============================================================
# PARSE TOP-DOWN
# Format: SKU | Month | Top
# Month can be: Jan, Jan-26, January, 2026-01, 2026-1, etc.
# No channel column — bridge will split across channels statistically.
# ============================================================
td_raw = pd.read_csv(td_filename)
print(f"Top-down raw: {len(td_raw)} rows")
print(f"Columns: {list(td_raw.columns)}")
print(td_raw.head(10).to_string())

# Detect column names flexibly
rename_td = {}
for c in td_raw.columns:
    cl = c.lower().strip().replace('-','').replace('_','').replace(' ','')
    if cl in ('sku','code','productcode','finishedgood'):
        rename_td[c] = 'sku'
    elif cl in ('month','period','monthyear','date','mon'):
        rename_td[c] = 'month_raw'
    elif cl in ('top','topdown','forecast','units','qty','quantity'):
        rename_td[c] = 'top_down'

td = td_raw.rename(columns=rename_td)

# Verify we have the three needed columns
missing = [c for c in ('sku','month_raw','top_down') if c not in td.columns]
if missing:
    print(f"\n⚠️  Could not auto-detect columns: {missing}")
    print("Available columns:", list(td.columns))
    print("Please rename your CSV columns to: SKU, Month, Top")
    raise ValueError(f"Missing columns: {missing}")

td = td[['sku','month_raw','top_down']].copy()
td['top_down'] = pd.to_numeric(
    td['top_down'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
).fillna(0).astype(int)  # strip thousands-separator commas
td['period']   = td['month_raw'].apply(parse_period)

failed = td[td['period'].isna()]
if len(failed):
    print(f"\n⚠️  Could not parse {len(failed)} month values: {failed['month_raw'].unique()}")

td = td.dropna(subset=['period'])
td = td[td['top_down'] >= 0].copy()  # keep 0s so bridge shows TD=0 gap explicitly
td['month_str']  = td['period'].astype(str)           # 'YYYY-MM'
td['month_year'] = td['period'].apply(period_to_my)   # 'Feb-26'

print(f"\nTop-down parsed: {len(td)} rows")
print(f"  SKUs  : {td['sku'].nunique()}")
print(f"  Months: {sorted(td['month_str'].unique())}")
print(f"  Total units: {td['top_down'].sum():,}")
print()
print(td[['sku','month_raw','month_str','month_year','top_down']].to_string(index=False))

# ── Debug: SKUs in BU Feb missing from TD Feb ────────────────────────────
# Run after BU is parsed (Cell 7) to catch silently dropped SKUs
try:
    bu_feb = bu[bu['month_str'] == '2026-02']['sku'].unique()
    td_feb = td[td['month_str'] == '2026-02']['sku'].unique()
    missing_from_td = set(bu_feb) - set(td_feb)
    print(f"\n⚠️  SKUs in BU Feb but missing from TD Feb: {len(missing_from_td)}")
    for s in sorted(missing_from_td):
        print(f"  {s} — {name_lookup.get(s, '?')}")
    if not missing_from_td:
        print("✅  All BU Feb SKUs are present in TD Feb")
except NameError:
    print("ℹ️  Run Cell 7 (BU parse) before this debug block")


Top-down raw: 708 rows
Columns: ['SKU', 'Month', 'Top']
        SKU Month  Top
0  FG-10094   Jan   90
1  FG-10094   Feb   90
2  FG-10094   Mar   80
3  FG-10094   Apr   90
4  FG-10094   May   65
5  FG-10094   Jun   75
6  FG-10094   Jul   70
7  FG-10094   Aug  135
8  FG-10094   Sep   80
9  FG-10094   Oct  200

Top-down parsed: 708 rows
  SKUs  : 59
  Months: ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09', '2026-10', '2026-11', '2026-12']
  Total units: 161,181

       sku month_raw month_str     month_year  top_down
  FG-10094       Jan   2026-01   January 2026        90
  FG-10094       Feb   2026-02  February 2026        90
  FG-10094       Mar   2026-03     March 2026        80
  FG-10094       Apr   2026-04     April 2026        90
  FG-10094       May   2026-05       May 2026        65
  FG-10094       Jun   2026-06      June 2026        75
  FG-10094       Jul   2026-07      July 2026        70
  FG-10094       Aug   2026-08    Au

In [ ]:
# ============================================================
# LOAD SALES HISTORY + BUILD SHARE STABILITY MODEL
# ============================================================
sales = pd.read_csv(sales_filename)
sales['created_date'] = pd.to_datetime(sales['created_date'])
sales['year_month_str'] = sales['created_date'].dt.to_period('M').astype(str)
sales['customers__customer_category'] = (
    sales['customers__customer_category'].fillna('No Customer Type'))

# Supplement name_lookup from sales
sales_names = (sales.groupby('products__variants__sku')
               ['products__root_product__title'].first()
               .fillna(sales.groupby('products__variants__sku')
               ['products__variants__sku'].first()).to_dict())
name_lookup = {**sales_names, **name_lookup}   # bottom-up names take priority

# ── Filter to share window ────────────────────────────────────────────────
cutoff_p     = pd.Period(SPLIT_CUTOFF, freq='M')
window_start = str(cutoff_p - (SHARE_MONTHS - 1))
break_cutoff = str(cutoff_p - (BREAK_RECENT_MONTHS - 1))

hist = sales[
    (sales['year_month_str'] >= window_start) &
    (sales['year_month_str'] <= SPLIT_CUTOFF) &
    (sales['orders__source'].isin(EXPORT_CHANNELS))
].copy()
hist = hist.rename(columns={'products__variants__sku':'sku',
                             'orders__source':'channel',
                             'customers__customer_category':'category'})

# Monthly units per SKU × channel × category
monthly_cat = (hist.groupby(['sku','channel','category','year_month_str'],
                             observed=True)['quantity'].sum().reset_index()
               .rename(columns={'year_month_str':'month','quantity':'units'}))

# Monthly total per SKU × channel (denominator)
monthly_ch_total = (monthly_cat.groupby(['sku','channel','month'])['units']
                    .sum().reset_index().rename(columns={'units':'ch_total'}))
monthly_cat = monthly_cat.merge(monthly_ch_total, on=['sku','channel','month'])
monthly_cat['share'] = np.where(monthly_cat['ch_total']>0,
    monthly_cat['units']/monthly_cat['ch_total'], 0)

# ── Monthly total per SKU (denominator for channel share) ─────────────────
monthly_sku_total = (monthly_cat.groupby(['sku','month'])['units']
                     .sum().reset_index().rename(columns={'units':'sku_total'}))
monthly_ch_only = (monthly_cat.groupby(['sku','channel','month'])['units']
                   .sum().reset_index().rename(columns={'units':'ch_units'}))
monthly_ch_only = monthly_ch_only.merge(monthly_sku_total, on=['sku','month'])
monthly_ch_only['ch_share'] = np.where(monthly_ch_only['sku_total']>0,
    monthly_ch_only['ch_units']/monthly_ch_only['sku_total'], 0)

print(f"History window: {window_start} → {SPLIT_CUTOFF}")
print(f"  Rows: {len(monthly_cat):,}")
print(f"  SKUs: {monthly_cat['sku'].nunique()}")

# ── Stability function ────────────────────────────────────────────────────
def stability_stats(grp, share_col='share'):
    shares = grp[share_col].values
    n = len(shares)
    mean_s = shares.mean()
    std_s  = shares.std(ddof=1) if n>1 else 0.0
    cv     = std_s/mean_s if mean_s>0 else 0.0
    if n >= 4:
        slope,_,_,pval,_ = stats.linregress(np.arange(n,dtype=float), shares)
    else:
        slope, pval = 0.0, 1.0
    has_trend = (abs(slope)>=TREND_SLOPE_MIN) and (pval<=TREND_PVALUE_MAX)
    recent = grp[grp['month']>=break_cutoff][share_col].values
    prior  = grp[grp['month']< break_cutoff][share_col].values
    rm = recent.mean() if len(recent)>0 else mean_s
    pm = prior.mean()  if len(prior)>0  else mean_s
    bd = abs(rm - pm)
    has_break = bd > BREAK_DIFF_MAX
    if   cv<=CV_STABLE_MAX  and not has_trend and not has_break: status='Stable'
    elif cv<=CV_MONITOR_MAX or  has_trend or has_break:          status='Monitor'
    else:                                                         status='Unstable'
    return dict(n_months=n, mean_share=mean_s, std_share=std_s, cv=cv,
                trend_slope=slope, trend_pvalue=pval, has_trend=has_trend,
                recent_mean=rm, prior_mean=pm, break_diff=bd, has_break=has_break,
                status=status)

# ── Category-level stability ──────────────────────────────────────────────
cat_records = []
for (sku,ch,cat), grp in monthly_cat.groupby(['sku','channel','category']):
    rec = stability_stats(grp, 'share')
    rec.update({'sku':sku,'channel':ch,'category':cat})
    cat_records.append(rec)
stability_df = pd.DataFrame(cat_records)

# n_categories per SKU × channel (for equal-weight fallback)
n_cats = (stability_df.groupby(['sku','channel'])['category'].nunique()
          .reset_index().rename(columns={'category':'n_cats'}))
stability_df = stability_df.merge(n_cats, on=['sku','channel'])

def alloc_share(row):
    eq = 1.0/row['n_cats'] if row['n_cats']>0 else 1.0
    if row['status']=='Stable':   return row['mean_share']
    elif row['status']=='Monitor': return MONITOR_BLEND_WEIGHT*row['mean_share']+(1-MONITOR_BLEND_WEIGHT)*eq
    else:                          return eq

stability_df['alloc_share_raw'] = stability_df.apply(alloc_share, axis=1)
alloc_totals = (stability_df.groupby(['sku','channel'])['alloc_share_raw']
                .sum().reset_index().rename(columns={'alloc_share_raw':'alloc_total'}))
stability_df = stability_df.merge(alloc_totals, on=['sku','channel'])
stability_df['alloc_share'] = np.where(stability_df['alloc_total']>0,
    stability_df['alloc_share_raw']/stability_df['alloc_total'], 0)

# ── Channel-level stability ───────────────────────────────────────────────
ch_records = []
n_ch_per_sku = (monthly_ch_only.groupby('sku')['channel'].nunique()
                .reset_index().rename(columns={'channel':'n_channels'}))
for (sku,ch), grp in monthly_ch_only.groupby(['sku','channel']):
    rec = stability_stats(grp, 'ch_share')
    n_ch = n_ch_per_sku[n_ch_per_sku['sku']==sku]['n_channels'].values
    n_ch = int(n_ch[0]) if len(n_ch) else len(EXPORT_CHANNELS)
    eq = 1.0/n_ch
    if rec['status']=='Stable':   ac = rec['mean_share']
    elif rec['status']=='Monitor': ac = MONITOR_BLEND_WEIGHT*rec['mean_share']+(1-MONITOR_BLEND_WEIGHT)*eq
    else:                          ac = eq
    rec.update({'sku':sku,'channel':ch,'n_channels':n_ch,'alloc_ch_share_raw':ac})
    ch_records.append(rec)
ch_stab_df = pd.DataFrame(ch_records)
ch_totals = (ch_stab_df.groupby('sku')['alloc_ch_share_raw'].sum()
             .reset_index().rename(columns={'alloc_ch_share_raw':'ch_alloc_total'}))
ch_stab_df = ch_stab_df.merge(ch_totals, on='sku')
ch_stab_df['alloc_ch_share'] = np.where(ch_stab_df['ch_alloc_total']>0,
    ch_stab_df['alloc_ch_share_raw']/ch_stab_df['ch_alloc_total'], 0)

counts = stability_df['status'].value_counts()
print(f"\nCategory stability: {len(stability_df)} buckets")
for s in ['Stable','Monitor','Unstable']:
    print(f"  {s:<10}: {counts.get(s,0)}")
ch_counts = ch_stab_df['status'].value_counts()
print(f"\nChannel stability: {len(ch_stab_df)} buckets")
for s in ['Stable','Monitor','Unstable']:
    print(f"  {s:<10}: {ch_counts.get(s,0)}")

History window: 2025-03 → 2026-02
  Rows: 2,119
  SKUs: 169

Category stability: 353 buckets
  Stable    : 249
  Monitor   : 70
  Unstable  : 34

Channel stability: 229 buckets
  Stable    : 157
  Monitor   : 60
  Unstable  : 12


In [ ]:
# ============================================================
# APPLY TOP-DOWN ALLOCATION
# SKU × month total → split by channel share → split by category share
# Integer remainder always to largest bucket
# ============================================================

def split_proportionally(total, shares_series):
    """Split integer total into parts using shares. Last bucket gets remainder."""
    parts = []
    allocated = 0
    shares = shares_series.values
    for i, s in enumerate(shares):
        if i < len(shares)-1:
            v = int(round(total * s))
        else:
            v = total - allocated
        allocated += v
        parts.append(v)
    return parts

export_rows = []

for _, td_row in td.iterrows():
    sku        = td_row['sku']
    month_str  = td_row['month_str']   # 'YYYY-MM'
    month_year = td_row['month_year']  # 'Feb-26'
    td_total   = int(td_row['top_down'])
    pname      = name_lookup.get(sku, sku)

    # Use the period object directly for robust month and year extraction
    p_obj = td_row['period']
    yr_val = p_obj.year
    mn_val = p_obj.month

    month_name = MONTH_NAMES[f'{mn_val:02d}'] # Format month as 2 digits for dict lookup
    month_num  = f"{yr_val}{mn_val}"           # e.g. 20267 for July 2026

    # ── Channel split ─────────────────────────────────────────────────────
    sku_ch = ch_stab_df[ch_stab_df['sku']==sku].sort_values(
        'alloc_ch_share', ascending=False).reset_index(drop=True)

    if len(sku_ch)==0:
        # No history — equal split
        sku_ch = pd.DataFrame([
            {'sku':sku,'channel':ch,'alloc_ch_share':1/len(EXPORT_CHANNELS),
             'status':'Unstable'} for ch in EXPORT_CHANNELS])

    ch_parts = split_proportionally(td_total, sku_ch['alloc_ch_share'])

    for (_, ch_row), ch_units in zip(sku_ch.iterrows(), ch_parts):
        ch = ch_row['channel']
        if ch_units == 0:
            # Don't silently skip — log it so the bridge surfaces the gap
            print(f"  ℹ️  {sku} {month_year} {ch}: ch_units rounded to 0 (td_total={td_total}), skipping channel")
            continue

        # ── Category split ────────────────────────────────────────────────
        sku_cats = stability_df[
            (stability_df['sku']==sku) & (stability_df['channel']==ch)
        ].sort_values('alloc_share', ascending=False).reset_index(drop=True)

        if len(sku_cats)==0:
            sku_cats = pd.DataFrame([{
                'sku':sku,'channel':ch,'category':'Unknown',
                'alloc_share':1.0,'status':'Unstable'}])

        cat_parts = split_proportionally(ch_units, sku_cats['alloc_share'])

        for (_, cat_row), cat_units in zip(sku_cats.iterrows(), cat_parts):
            composite_key = f"{pname}-{sku}-{ch}-{month_year}"  # e.g. "...FG-10004-Wholesale-July 2026"
            export_rows.append({
                'composite_primary_key':        composite_key,
                'product_name':                 pname,
                'code':                         sku,
                'channel':                      ch,
                'customers__customer_category': cat_row['category'],
                'month_num':                    month_num,
                'month_year':                   month_year,
                'units':                        cat_units,
                '_td_total':                    td_total,
                '_ch_status':                   ch_row['status'],
                '_cat_status':                  cat_row['status'],
            })

export_df = pd.DataFrame(export_rows)
print(f"\u2705 Allocation complete: {len(export_df):,} export rows")

if len(export_df):
    # Verify sums
    check = export_df.groupby(['code','month_year'])['units'].sum().reset_index()
    check = check.merge(td[['sku','month_year','top_down']].rename(columns={'sku':'code'}),
                        on=['code','month_year'], how='left')
    bad = check[check['units'] != check['top_down']]
    if len(bad):
        print(f"\u26a0\ufe0f  {len(bad)} rows where sum \u2260 top-down total (rounding edge cases):")
        print(bad.head())
    else:
        print("\u2705  All totals match top-down exactly")

    # Sample
    sample_sku = export_df['code'].iloc[0]
    sample_my  = export_df['month_year'].iloc[0]
    chk = export_df[(export_df['code']==sample_sku)&(export_df['month_year']==sample_my)]
    print(f"\nSample — {sample_sku} {sample_my}:")
    print(chk[['channel','customers__customer_category','units','_ch_status','_cat_status']].to_string(index=False))
    print(f"  Sum: {chk['units'].sum()}  |  Top-down: {chk['_td_total'].iloc[0]}")

✅ Allocation complete: 2,340 export rows
✅  All totals match top-down exactly

Sample — FG-10094 January 2026:
           channel customers__customer_category  units _ch_status _cat_status
Direct-to-Consumer           Direct-to-Consumer     52     Stable      Stable
         Wholesale   Professional - Esthetician     23    Monitor      Stable
         Wholesale         Retailer - Non-Credo     15    Monitor    Unstable
  Sum: 90  |  Top-down: 90


In [ ]:
# ============================================================
# BUILD BRIDGE: Bottom-Up vs Top-Down vs Difference
# Grain: SKU × channel × month
# bu already at SKU × channel × month grain (no category column)
# ============================================================

# Bottom-up is already SKU × channel × month — just use it directly
bu_agg = bu[['sku','product_name','channel','month_str','month_year','bu_units']].copy()

# Top-down: aggregate export_df across categories → SKU × channel × month
td_agg = (export_df.groupby(['code','channel','month_year'])['units']
           .sum().reset_index()
           .rename(columns={'code':'sku','units':'td_units'}))

bridge = bu_agg.merge(td_agg, on=['sku','channel','month_year'], how='outer')
bridge['bu_units']   = bridge['bu_units'].fillna(0).astype(int)
bridge['td_units']   = bridge['td_units'].fillna(0).astype(int)
bridge['difference'] = bridge['td_units'] - bridge['bu_units']
bridge['diff_pct']   = np.where(bridge['bu_units'] != 0,
    (bridge['difference'] / bridge['bu_units'] * 100).round(1), np.nan)
bridge['product_name'] = bridge['product_name'].fillna(
    bridge['sku'].map(name_lookup))
bridge = bridge.sort_values(['sku','channel','month_str']).reset_index(drop=True)

print(f"Bridge rows: {len(bridge):,}")
sample_sku = bridge['sku'].iloc[0]
print(f"\nSample — {sample_sku}:")
print(bridge[bridge['sku']==sample_sku][
    ['channel','month_year','bu_units','td_units','difference','diff_pct']
].to_string(index=False))
print()
print(f"Totals:")
print(f"  Bottom-up  : {bridge['bu_units'].sum():,}")
print(f"  Top-down   : {bridge['td_units'].sum():,}")
print(f"  Difference : {bridge['difference'].sum():+,}")


Bridge rows: 3,115

Sample — FG-100026:
  channel     month_year  bu_units  td_units  difference  diff_pct
Wholesale     April 2026         0         5           5       NaN
Wholesale    August 2026         0         5           5       NaN
Wholesale  December 2026         0         5           5       NaN
Wholesale  February 2026         0         5           5       NaN
Wholesale   January 2026         0         5           5       NaN
Wholesale      July 2026         0         5           5       NaN
Wholesale      June 2026         0         5           5       NaN
Wholesale     March 2026         0         5           5       NaN
Wholesale       May 2026         0         5           5       NaN
Wholesale  November 2026         0         5           5       NaN
Wholesale   October 2026         0         5           5       NaN
Wholesale September 2026         0         5           5       NaN

Totals:
  Bottom-up  : 160,845
  Top-down   : 161,181
  Difference : +336


In [ ]:
print("Authenticating with Google...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("✅ Authenticated")

sheet_name = f"Forecast_Bridge_{datetime.now().strftime('%Y%m%d_%H%M')}"
sh = gc.create(sheet_name)   # ← this defines `sh`

Authenticating with Google...
✅ Authenticated


In [ ]:
# ── Share Stability tab ──────────────────────────────────────────────────
ws_stab = sh.sheet1
ws_stab.update_title('Share Stability')

stab_hdr = ['SKU','Product Name','Channel','Category','N Months',
            'Mean Share','CV','Trend Slope','Trend p-val','Has Trend',
            'Break Diff','Has Break','Status','Alloc Share']

# Helper function to safely round or return None for NaN
def safe_round(value, decimals):
    if pd.isna(value):
        return None
    return round(value, decimals)

stab_rows = []
for _, r in stability_df.sort_values(['sku','channel','status','category']).iterrows():
    stab_rows.append([
        r['sku'], name_lookup.get(r['sku'],r['sku']), r['channel'], r['category'],
        r['n_months'], safe_round(r['mean_share'],4), safe_round(r['cv'],4),
        safe_round(r['trend_slope'],6), safe_round(r['trend_pvalue'],4), str(r['has_trend']),
        safe_round(r['break_diff'],4), str(r['has_break']),
        r['status'], safe_round(r['alloc_share'],4)
    ])

note = [[f'Share stability diagnostics  |  Window: last {SHARE_MONTHS} months to {SPLIT_CUTOFF}  |  '
         f'Stable CV<{CV_STABLE_MAX}  Monitor CV<{CV_MONITOR_MAX}  else Unstable']]

# Write all data in one call
ws_stab.update('A1', note + [stab_hdr] + stab_rows)
ws_stab.freeze(rows=2)

# ── Batch all formatting ──────────────────────────────────────────────────
def bg(r,g,b): return {'red':r,'green':g,'blue':b}

def fmt_range(ws, r1, c1, r2, c2, fmt):
    return {'repeatCell': {
        'range': {'sheetId': ws.id,
                  'startRowIndex':r1,'endRowIndex':r2,
                  'startColumnIndex':c1,'endColumnIndex':c2},
        'cell': {'userEnteredFormat': fmt},
        'fields': 'userEnteredFormat(' + ','.join(fmt.keys()) + ')'
    }}

STATUS_BG = {
    'Stable':   bg(0.85, 0.95, 0.85),
    'Monitor':  bg(1.0,  0.97, 0.8),
    'Unstable': bg(1.0,  0.87, 0.87),
}

# Group rows by status for contiguous-range batching
from collections import defaultdict
status_rows = defaultdict(list)
for i, r in enumerate(stab_rows):
    status_rows[r[12]].append(i + 2)   # +2: note row + header row (0-indexed)

def contiguous_ranges(rows):
    if not rows: return []
    ranges = []; start = rows[0]; prev = rows[0]
    for r in rows[1:]:
        if r == prev+1: prev = r
        else:
            ranges.append((start, prev)); start=r; prev=r
    ranges.append((start, prev))
    return ranges

fmt_requests = [
    # Note row
    fmt_range(ws_stab, 0, 0, 1, 14, {
        'textFormat': {'bold':True,'italic':True,
                       'foregroundColor': bg(0.4,0.2,0.0)},
        'backgroundColor': bg(1.0,0.95,0.8)
    }),
    # Header row
    fmt_range(ws_stab, 1, 0, 2, 14, {
        'textFormat': {'bold':True,'foregroundColor': bg(1,1,1)},
        'backgroundColor': bg(0.18,0.33,0.53)
    }),
]

for status, rows in status_rows.items():
    colour = STATUS_BG.get(status)
    if not colour: continue
    for start, end in contiguous_ranges(sorted(rows)):
        fmt_requests.append(fmt_range(ws_stab, start, 12, end+1, 13, {
            'backgroundColor': colour,
            'textFormat': {'bold': True}
        }))

sh.batch_update({'requests': fmt_requests})

print(f"\u2705 Share Stability: {len(stab_rows)} rows")
for s in ['Stable','Monitor','Unstable']:
    print(f"  {s:<10}: {len(status_rows[s])}")

✅ Share Stability: 353 rows
  Stable    : 249
  Monitor   : 70
  Unstable  : 34


In [ ]:
# ── Bridge tab ───────────────────────────────────────────────────────────
ws_bridge = sh.add_worksheet(title='Bridge', rows=5000, cols=10)

bridge_hdr = ['SKU','Product Name','Channel','Month','Month Year',
              'Bottom-Up','Top-Down','Difference','Diff %']
bridge_rows = []
for _, r in bridge.iterrows():
    bridge_rows.append([
        r['sku'], r.get('product_name',''), r['channel'],
        str(r['month_str']) if pd.notna(r.get('month_str')) else '',
        r['month_year'],
        int(r['bu_units']), int(r['td_units']), int(r['difference']),
        round(r['diff_pct'],1) if pd.notna(r['diff_pct']) else ''
    ])

# Write all data in one call
ws_bridge.update('A1', [bridge_hdr] + bridge_rows)

# ── Batch all formatting into a single API call ───────────────────────────
fmt_requests = []

def bg(red, green, blue):
    return {'red': red, 'green': green, 'blue': blue}

def fmt_range(ws, r1, c1, r2, c2, fmt):
    return {
        'repeatCell': {
            'range': {
                'sheetId': ws.id,
                'startRowIndex': r1, 'endRowIndex': r2,
                'startColumnIndex': c1, 'endColumnIndex': c2,
            },
            'cell': {'userEnteredFormat': fmt},
            'fields': 'userEnteredFormat(' + ','.join(fmt.keys()) + ')'
        }
    }

# Header row (row 0)
fmt_requests.append(fmt_range(ws_bridge, 0, 0, 1, 9, {
    'textFormat': {'bold': True, 'foregroundColor': bg(1,1,1)},
    'backgroundColor': bg(0.259, 0.522, 0.957)
}))
# Bottom-Up header — slate
fmt_requests.append(fmt_range(ws_bridge, 0, 5, 1, 6, {
    'backgroundColor': bg(0.36, 0.44, 0.56)
}))
# Top-Down header — green
fmt_requests.append(fmt_range(ws_bridge, 0, 6, 1, 7, {
    'backgroundColor': bg(0.18, 0.53, 0.33)
}))
# Difference header — amber
fmt_requests.append(fmt_range(ws_bridge, 0, 7, 1, 9, {
    'backgroundColor': bg(0.85, 0.65, 0.13)
}))

# Collect positive and negative diff rows, batch them in groups
pos_rows = []
neg_rows = []
for i, r in enumerate(bridge_rows):
    diff = r[7]
    if   diff > 0: pos_rows.append(i + 1)   # +1 for header offset (0-indexed)
    elif diff < 0: neg_rows.append(i + 1)

# Add one format request per contiguous range (reduces API calls massively)
def contiguous_ranges(rows):
    if not rows: return []
    ranges = []
    start = rows[0]; prev = rows[0]
    for r in rows[1:]:
        if r == prev + 1:
            prev = r
        else:
            ranges.append((start, prev))
            start = r; prev = r
    ranges.append((start, prev))
    return ranges

for start, end in contiguous_ranges(pos_rows):
    fmt_requests.append(fmt_range(ws_bridge, start, 7, end+1, 8, {
        'backgroundColor': bg(0.85, 0.95, 0.85)
    }))
for start, end in contiguous_ranges(neg_rows):
    fmt_requests.append(fmt_range(ws_bridge, start, 7, end+1, 8, {
        'backgroundColor': bg(1.0, 0.87, 0.87)
    }))

# Execute all formatting in ONE batch request
sh.batch_update({'requests': fmt_requests})
ws_bridge.freeze(rows=1)

print(f"\u2705 Bridge: {len(bridge_rows)} rows  |  "
      f"{len(pos_rows)} positive (green)  |  {len(neg_rows)} negative (red)")


✅ Bridge: 3115 rows  |  1116 positive (green)  |  1828 negative (red)


In [ ]:
# ── Export - Top-Down Split tab ──────────────────────────────────────────
ws_exp = sh.add_worksheet(title='Export - Top-Down Split', rows=20000, cols=9)

EXPORT_COLS = ['composite_primary_key','product_name','code','channel',
               'customers__customer_category','month_num','month_year','units']

exp_rows = export_df[EXPORT_COLS].values.tolist()
ws_exp.update('A1', [EXPORT_COLS] + exp_rows)
ws_exp.format('A1:H1', {'textFormat':{'bold':True,'foregroundColor':{'red':1,'green':1,'blue':1}},
    'backgroundColor':{'red':0.18,'green':0.53,'blue':0.33}})
ws_exp.freeze(rows=1)

print(f"\u2705 Export - Top-Down Split: {len(exp_rows):,} rows")
print()
print("="*70)
print("FORECAST BRIDGE COMPLETE")
print("="*70)
print(f"URL: https://docs.google.com/spreadsheets/d/{sh.id}")
print()
for ws in sh.worksheets():
    print(f"  - {ws.title}")
print()
bu_total  = bridge['bu_units'].sum()
td_total  = bridge['td_units'].sum()
print(f"Bottom-up total  : {bu_total:>10,}")
print(f"Top-down total   : {td_total:>10,}")
print(f"Net difference   : {td_total-bu_total:>+10,}")
stable_pct = len(stability_df[stability_df['status']=='Stable'])/len(stability_df)*100
print(f"\nShare buckets stable: {stable_pct:.0f}%  "
      f"monitor: {len(stability_df[stability_df['status']=='Monitor'])}  "
      f"unstable: {len(stability_df[stability_df['status']=='Unstable'])}")

✅ Export - Top-Down Split: 2,340 rows

FORECAST BRIDGE COMPLETE
URL: https://docs.google.com/spreadsheets/d/1vDo5ZXnP8sXw6HYwgf4NvShgQAylCLkWnzfdX5Sewmg

  - Share Stability
  - Bridge
  - Export - Top-Down Split

Bottom-up total  :    160,845
Top-down total   :    161,181
Net difference   :       +336

Share buckets stable: 71%  monitor: 70  unstable: 34
